# Data 模块教程

本教程介绍 open-xquant 数据层的核心用法：**下载市场数据**、**读取市场数据**、以及**下载和读取宏观因子数据**。

数据层采用 **Download-then-Read** 架构：
- **Downloader** 负责从外部 API 获取数据，保存为本地 Parquet 文件
- **LocalMarketDataProvider** 负责从本地 Parquet 文件读取行情数据，提供给策略管道
- **WorldBankDownloader** 负责下载宏观经济指标（GDP、CPI 等），**read_factor()** 负责读取

这种分离确保了策略执行的确定性——策略管道永远不会直接访问网络。

## 1. 安装依赖

open-xquant 的数据源作为可选依赖安装：

```bash
# 美股/全球市场（yfinance）
pip install open-xquant[yfinance]

# A 股市场（akshare）
pip install open-xquant[akshare]

# 两者都安装
pip install open-xquant[yfinance,akshare]
```

---
## 2. 下载美股数据（YFinance）

使用 `YFinanceDownloader` 从 Yahoo Finance 下载 OHLCV 日线数据。

In [1]:
from oxq.data import YFinanceDownloader

downloader = YFinanceDownloader()

# 下载 Apple 股票 2024 年全年数据
path = downloader.download("AAPL", start="2024-01-01", end="2024-12-31")
print(f"数据已保存到: {path}")

数据已保存到: /Users/daodao/.oxq/data/market/AAPL.parquet


下载完成后，数据以 Parquet 格式存储在本地。我们可以直接读取验证：

In [2]:
import pandas as pd

df = pd.read_parquet(path)
print(f"数据形状: {df.shape}")
print(f"时间范围: {df.index[0].date()} ~ {df.index[-1].date()}")
print(f"列: {list(df.columns)}")
print()
df.head()

数据形状: (251, 5)
时间范围: 2024-01-02 ~ 2024-12-30
列: ['open', 'high', 'low', 'close', 'volume']



,open,high,low,close,volume
date,,,,,
2024-01-02,185.225762,186.502507,181.999286,183.731293,82488700
2024-01-03,182.325916,183.968852,181.544030,182.355606,58414500
2024-01-04,180.277196,181.207533,179.020264,180.039673,71983600
2024-01-05,180.118838,180.880911,178.317544,179.317154,62379700
2024-01-08,180.217791,183.691712,179.633861,183.652115,59144500


所有 Downloader 输出统一的 DataFrame 格式：
- **Index**: `DatetimeIndex`，名称为 `date`
- **Columns**: `open`, `high`, `low`, `close`, `volume`（全小写）

---
## 3. 下载 A 股数据（AkShare）

使用 `AkShareDownloader` 下载 A 股日线数据。注意 A 股 symbol 使用纯数字代码（如 `600519`）。

In [3]:
from oxq.data import AkShareDownloader

ak_downloader = AkShareDownloader()

# 下载贵州茅台 2024 年数据（前复权）
path_cn = ak_downloader.download("600519", start="20240101", end="20241231")
print(f"数据已保存到: {path_cn}")

ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))

In [4]:
df_cn = pd.read_parquet(path_cn)
print(f"数据形状: {df_cn.shape}")
print(f"时间范围: {df_cn.index[0].date()} ~ {df_cn.index[-1].date()}")
print()
df_cn.head()

NameError: name 'path_cn' is not defined

无论数据源是 yfinance 还是 akshare，输出格式完全一致。这是框架的核心设计原则——**统一的数据接口**。

---
## 4. 读取数据（LocalMarketDataProvider）

下载完成后，策略通过 `LocalMarketDataProvider` 读取本地数据。这是策略管道获取行情数据的**唯一入口**。

In [5]:
from oxq.data import LocalMarketDataProvider

provider = LocalMarketDataProvider()

# 读取指定时间范围的数据
bars = provider.get_bars("AAPL", start="2024-06-01", end="2024-06-30")
print(f"6 月数据: {len(bars)} 个交易日")
bars

6 月数据: 19 个交易日


,open,high,low,close,volume
date,,,,,
2024-06-03,191.419546,193.493517,191.042473,192.540878,50080500
2024-06-04,193.146197,193.820986,191.548552,192.858429,47471400
2024-06-05,193.900381,195.388869,193.374450,194.366776,54156800
2024-06-06,194.188147,194.991928,192.679808,192.987427,41181800
2024-06-07,193.156138,195.428571,192.650057,195.378952,53103900
2024-06-10,195.388867,195.785807,190.675322,191.637878,97010200
2024-06-11,192.163803,205.570129,192.143968,205.560196,172373300
2024-06-12,205.778517,218.510054,205.312123,211.434784,198134300
2024-06-13,213.091942,215.086510,209.976041,212.595779,97862700


`get_latest()` 返回最近一个交易日的数据（`pd.Series`）：

In [6]:
latest = provider.get_latest("AAPL")
print(f"最新收盘价: {latest['close']:.2f}")
print(f"最新成交量: {int(latest['volume']):,}")
print()
latest

最新收盘价: 250.83
最新成交量: 35,557,500



open      2.508596e+02
high      2.521227e+02
low       2.493877e+02
close     2.508298e+02
volume    3.555750e+07
Name: 2024-12-30 00:00:00, dtype: float64

---
## 5. 自定义数据目录

默认情况下，数据存储在 `~/.oxq/data/market/`。有三种方式自定义路径：

### 方式一：构造参数

In [7]:
from pathlib import Path
import tempfile

# 使用临时目录演示
custom_dir = Path(tempfile.mkdtemp()) / "my_data"

# Downloader 和 Provider 都支持自定义目录
downloader.download("MSFT", "2024-01-01", "2024-03-31", dest_dir=custom_dir)

custom_provider = LocalMarketDataProvider(data_dir=custom_dir)
msft = custom_provider.get_bars("MSFT", "2024-01-01", "2024-03-31")
print(f"自定义目录读取成功: {len(msft)} 条数据")
print(f"数据目录: {custom_dir}")

自定义目录读取成功: 61 条数据
数据目录: /var/folders/v_/vs3gpcts6vn_gtf7nkpgbffm0000gn/T/tmpu51tmla1/my_data


### 方式二：环境变量

设置 `OXQ_DATA_DIR` 环境变量，所有组件自动使用 `$OXQ_DATA_DIR/market/` 作为数据目录：

```bash
export OXQ_DATA_DIR=/path/to/my/data
```

### 优先级

构造参数 > 环境变量 > 默认值 (`~/.oxq/data/market/`)

---
## 6. 错误处理

数据层提供明确的异常类型，方便调用方处理。

In [8]:
from oxq.core import SymbolNotFoundError, DownloadError

# 读取未下载的标的 → SymbolNotFoundError
try:
    provider.get_bars("UNKNOWN_SYMBOL", "2024-01-01", "2024-12-31")
except SymbolNotFoundError as e:
    print(f"捕获异常: {e}")

捕获异常: No data for 'UNKNOWN_SYMBOL'. Run downloader first.


In [9]:
# 下载不存在的标的 → DownloadError
try:
    downloader.download("INVALID_TICKER_XYZ", "2024-01-01", "2024-12-31")
except DownloadError as e:
    print(f"捕获异常: {e}")

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: INVALID_TICKER_XYZ"}}}
$INVALID_TICKER_XYZ: possibly delisted; no timezone found


捕获异常: No data returned for 'INVALID_TICKER_XYZ' (2024-01-01 to 2024-12-31).


两种异常都继承自 `OxqError`，可以统一捕获：

```python
from oxq.core import OxqError

try:
    ...
except OxqError as e:
    print(f"open-xquant 错误: {e}")
```

---
## 7. 批量下载

使用 `download_many()` 一次下载多个标的：

In [10]:
symbols = ["GOOGL", "AMZN", "META"]
paths = downloader.download_many(symbols, start="2024-01-01", end="2024-12-31")

for symbol, path in paths.items():
    df = pd.read_parquet(path)
    print(f"{symbol}: {len(df)} 个交易日, 最新收盘价 {df['close'].iloc[-1]:.2f}")

GOOGL: 251 个交易日, 最新收盘价 190.36
AMZN: 251 个交易日, 最新收盘价 221.30
META: 251 个交易日, 最新收盘价 588.88


下载完成后，可以通过 `LocalMarketDataProvider` 统一读取所有标的：

In [11]:
for symbol in symbols:
    latest = provider.get_latest(symbol)
    print(f"{symbol}: close={latest['close']:.2f}, volume={latest['volume']:,}")

GOOGL: close=190.36, volume=14,264,700.0
AMZN: close=221.30, volume=28,321,200.0
META: close=588.88, volume=7,025,900.0


---
## 8. 宏观因子数据（World Bank）

除了标的行情数据，数据层还支持下载**宏观经济指标**。框架将所有外部数据统一视为 Factor（因子）——GDP、CPI 与 RSI 在策略视角下本质相同，都是"某个时间点上的一个数值"。

当前支持 4 个 World Bank 指标：

| 名称 | 说明 |
|------|------|
| `gdp` | GDP（current USD） |
| `gdp_per_capita` | 人均 GDP（current USD） |
| `gdp_growth` | GDP 增长率（annual %） |
| `cpi` | CPI 通胀率（annual %） |

In [12]:
from oxq.data import WorldBankDownloader

wb = WorldBankDownloader()

# 下载中国和美国 2015-2023 年的 GDP 数据
path_gdp = wb.download("gdp", countries=["CHN", "USA"], start_year=2015, end_year=2023)
print(f"GDP 数据已保存到: {path_gdp}")

GDP 数据已保存到: /Users/daodao/.oxq/data/factor/gdp.parquet


因子数据以 Parquet 格式存储在 `~/.oxq/data/factor/` 目录下。使用 `read_factor()` 读取本地数据：

In [13]:
from oxq.data import read_factor

# 读取刚下载的 GDP 数据
gdp = read_factor("gdp")
print(f"年份范围: {gdp.index[0]} ~ {gdp.index[-1]}")
print(f"国家: {list(gdp.columns)}")
print()
gdp

年份范围: 2015 ~ 2023
国家: ['CHN', 'USA']



,CHN,USA
year,,
2015,1.128081e+13,1.820602e+13
2016,1.145602e+13,1.869511e+13
2017,1.253756e+13,1.947734e+13
2018,1.414777e+13,2.053306e+13
2019,1.456017e+13,2.138098e+13
2020,1.499641e+13,2.106047e+13
2021,1.820170e+13,2.331508e+13
2022,1.831677e+13,2.560485e+13
2023,1.827036e+13,2.729217e+13


`read_factor()` 支持按国家和年份筛选：

In [14]:
# 只看中国，2020 年以后
gdp_chn = read_factor("gdp", countries=["CHN"], start_year=2020)
print("中国 GDP (2020-):")
print(gdp_chn)
print()

# 只看美国，2018-2021 年
gdp_usa = read_factor("gdp", countries=["USA"], start_year=2018, end_year=2021)
print("美国 GDP (2018-2021):")
print(gdp_usa)

中国 GDP (2020-):
               CHN
year              
2020  1.499641e+13
2021  1.820170e+13
2022  1.831677e+13
2023  1.827036e+13

美国 GDP (2018-2021):
               USA
year              
2018  2.053306e+13
2019  2.138098e+13
2020  2.106047e+13
2021  2.331508e+13


下载其他指标同样简单——只需更换 `indicator` 参数：

In [15]:
# 下载 GDP 增长率和 CPI 通胀率
countries = ["CHN", "USA", "JPN", "DEU"]

wb.download("gdp_growth", countries=countries, start_year=2015, end_year=2023)
wb.download("cpi", countries=countries, start_year=2015, end_year=2023)

# 对比中美 GDP 增长率
growth = read_factor("gdp_growth", countries=["CHN", "USA"], start_year=2019)
print("中美 GDP 增长率 (%):")
print(growth)
print()

# 查看主要经济体 CPI
cpi = read_factor("cpi", start_year=2020)
print("主要经济体 CPI 通胀率 (%):")
print(cpi)

中美 GDP 增长率 (%):
           CHN       USA
year                    
2019  6.068502  2.583825
2020  2.340188 -2.163029
2021  8.570085  6.055053
2022  3.134189  2.512375
2023  5.414843  2.887556

主要经济体 CPI 通胀率 (%):
           CHN       DEU       JPN       USA
year                                        
2020  2.419422  0.144878 -0.024996  1.233584
2021  0.981015  3.066667 -0.233353  4.697859
2022  1.973576  6.872574  2.497703  8.002800
2023  0.234837  5.946437  3.268134  4.116338


---
## 小结

本教程覆盖了 data 模块的核心用法：

| 组件 | 职责 |
|------|------|
| `YFinanceDownloader` | 下载美股/全球市场数据 |
| `AkShareDownloader` | 下载 A 股数据 |
| `LocalMarketDataProvider` | 从本地 Parquet 读取数据 |
| `WorldBankDownloader` | 下载宏观指标（GDP、CPI 等） |
| `read_factor()` | 读取本地因子数据 |
| `resolve_data_dir()` | 解析市场数据目录路径 |
| `resolve_factor_dir()` | 解析因子数据目录路径 |

**数据存储结构**：

```
~/.oxq/data/
├── market/          # 行情数据（per symbol）
│   ├── AAPL.parquet
│   └── 600519.parquet
└── factor/          # 宏观因子（per indicator）
    ├── gdp.parquet
    ├── gdp_growth.parquet
    └── cpi.parquet
```

**核心设计原则**：
- 下载与读取分离，策略执行不依赖网络
- 统一的 DataFrame 格式（行情 OHLCV + DatetimeIndex，因子 year × country）
- 一切皆 Factor——宏观数据与技术指标在策略视角下本质相同
- Protocol 接口，支持自定义数据源扩展